# 01 EDA
Loads the Phase 1 outputs (`data/processed/*.parquet`, `reports/data_profile.json`) and plots them. No logic lives here.

In [ ]:
import json

import matplotlib.pyplot as plt
import pandas as pd

from reviewlens.config import load_config, resolve_path

cfg = load_config("data")
proc = resolve_path(cfg["paths"]["processed"])
splits = {k: pd.read_parquet(proc / f"{k}.parquet") for k in ["train", "val", "test", "timehold"]}
profile = json.loads((resolve_path(cfg["paths"]["reports"]) / "data_profile.json").read_text())
order = ["negative", "neutral", "positive"]

## Cleaning funnel

In [ ]:
pd.Series(profile["funnel"], name="rows").to_frame()

## Class balance per split

In [ ]:
share = pd.DataFrame({k: v["label"].value_counts(normalize=True) for k, v in splits.items()}).loc[order]
ax = share.T.plot.bar(stacked=True, figsize=(6, 3.5), rot=0)
ax.set_ylabel("share of reviews"); ax.set_title("Label share by split")
plt.tight_layout(); plt.show()
share.round(3)

## Review length (tokens, whitespace split)

In [ ]:
full = pd.concat(splits.values())
tokens = full["text"].str.split().str.len()
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
tokens.clip(upper=300).plot.hist(bins=60, ax=axes[0]); axes[0].set_title("Tokens per review (clipped at 300)")
full.assign(tokens=tokens).groupby("label")["tokens"].median().loc[order].plot.bar(ax=axes[1], rot=0)
axes[1].set_title("Median tokens by label")
plt.tight_layout(); plt.show()
profile["overall"]["tokens"], profile["overall"]["share_under_10_tokens"]

## Volume and negative share over time
The dashed line marks where the time-based hold-out starts.

In [ ]:
monthly = full.set_index("date").sort_index()
by_year = monthly.groupby(monthly.index.to_period("Y"))
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
by_year.size().plot(ax=axes[0], title="Reviews per year (in the sample)")
by_year["label"].apply(lambda s: (s == "negative").mean()).plot(ax=axes[1], title="Negative share per year")
for ax in axes:
    ax.axvline(pd.Period(profile["timehold_starts"][:4], "Y"), ls="--", c="gray")
plt.tight_layout(); plt.show()

## Stars vs label, and verified purchases

In [ ]:
print(pd.crosstab(full["rating"], full["label"]))
profile["overall"]["verified_purchase_share"]